# Jais Arabic Voice Chatbot — مباشرة على كولاب 🇸🇦🎙️

بيئة تشغيل جاهزة للتشغيل المباشر على **T4 (16 GB)**:

`Browser (WebRTC) → Jais-ASR (STT) → Jais-3 (LLM) → Audar-TTS (TTS)`

**الخطوات:**
1. `Runtime → Change runtime type → GPU (T4)`
2. `Run all` (أول مرة ≈ 10 دقائق: pip + ~7 GB نماذج)
3. الخلية 6 تفتح لك طرفًا جديدًا — اضغط **Connect** وابدأ الكلام بالعربية 🎙️

[&#8593; افتح هذا الدفتر في Colab ](https://colab.research.google.com/github/911houssem/jais-voice-chatbot/blob/main/notebooks/jais_voice_chatbot_colab.ipynb)

In [ ]:
#@title 1) فحص كرت الشاشة واختيار الجودة المناسبة {display-mode:"form"}
import subprocess, re

def gpu_state():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader"]).decode()
        name, total, free = [x.strip() for x in out.split(",")]
        return name, int(re.sub(r"\D", "", total)) / 1024.0, int(re.sub(r"\D", "", free)) / 1024.0
    except Exception as e:
        print("No GPU detected:", e)
        return "", 0.0, 0.0

name, total_gb, free_gb = gpu_state()
print(f"GPU: {name} | total {total_gb:.1f} GB | free {free_gb:.1f} GB")
if free_gb and free_gb < 20:
    print("Less than 20 GB free -> JAIS_ASR_MODEL=auto picks the W4A16-AWQ ASR checkpoint.")
else:
    print("20+ GB free -> JAIS_ASR_MODEL=auto picks the fp16 Jais-ASR checkpoint.")

In [ ]:
#@title 2) استنساخ المشروع وتثبيت الاعتمادات (~5 دقائق) {display-mode:"form"}
REPO_URL = "https://github.com/911houssem/jais-voice-chatbot.git" #@param {type:"string"}

import os, subprocess
repo_dir = "/content/jais-voice-chatbot"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
os.chdir(repo_dir)
print("cwd:", os.getcwd())

# Runner extra pulls the prebuilt WebRTC UI; compressed-tensors is needed by
# the W4A16-AWQ ASR checkpoint auto-selected on 16 GB GPUs.
!pip install -q -r requirements.txt
print("Install done.")

In [ ]:
#@title 3) (اختياري) رمز Hugging Face {display-mode:"form"}
import os
TOKEN = "" #@param {type:"string"}
# Not strictly required: all three repos are public. Set only if rate-limited.
if TOKEN:
    os.environ["HF_TOKEN"] = TOKEN
    from huggingface_hub import login
    login(TOKEN, add_to_git_credential=False)
else:
    print("No token set (fine for public repos).")

In [ ]:
#@title 4) تحميل النماذج مسبقًا (~7 GB — يتسارع أول اتصال) {display-mode:"form"}
from huggingface_hub import snapshot_download
for repo in ["IFM/Jais-ASR-W4A16-AWQ",        # STT quantized (T4 default)
             "inception42/Jais-3-0.9B",        # chat LLM
             "audarai/Audar-TTS-V1-Flash",     # TTS backbone + voices
             "neuphonic/neucodec"]:            # TTS codec
    print("downloading", repo, "...")
    snapshot_download(repo_id=repo)
print("Model cache is warm ✔")
# On a 20GB+ GPU the app swaps to IFM/Jais-ASR (fp16) instead, which downloads
# lazily on first connection.

In [ ]:
#@title 5) تشغيل خادم الصوت (WebRTC عبر TURN relay) {display-mode:"form"}
APP_PORT = 8080 #@param {type:"integer"}

import os, subprocess
# WebRTC media cannot enter the Colab VM over UDP, so both peers negotiate via
# these relay servers (free openrelay.metered.ca TURN over TCP). pipecat feeds
# PIPECAT_ICE_SERVERS to the bot AND to the browser through /start.
os.environ["JAIS_ASR_MODEL"] = "auto"          # AWQ checkpoint on T4 (~3.4 GB)
os.environ["JAIS_LLM_DTYPE"] = "fp16"
os.environ["JAIS_TTS_VOICE"] = "demo_female_1"
os.environ["PIPECAT_ICE_SERVERS"] = '[{"urls":["stun:stun.l.google.com:19302","turn:openrelay.metered.ca:80","turn:openrelay.metered.ca:443","turns:openrelay.metered.ca:443?transport=tcp"],"username":"openrelayproject","credential":"openrelayproject"}]'

logf = open("/content/jais-voice.log", "a")
proc = subprocess.Popen(
    ["python", "app.py", "-t", "webrtc", "--host", "0.0.0.0", "--port", str(APP_PORT)],
    stdout=logf, stderr=logf,
)
print("server started (PID %d)" % proc.pid)

In [ ]:
#@title 6) افتح واجهة الدردشة (نافذة جديدة) {display-mode:"form"}
import time, requests
from google.colab.output import eval_js, serve_kernel_port_as_window

for _ in range(90):
    try:
        if requests.get(f"http://127.0.0.1:{APP_PORT}/docs", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(2)
else:
    print("Uvicorn not up yet — check /content/jais-voice.log, then re-run this cell.")

proxy_base = eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
print("Browser URL:", proxy_base.rstrip('/') + "/client")
print("\nأو اضغط الزر أدناه لفتح طرف مباشر في /client:")
serve_kernel_port_as_window(port=APP_PORT)

## Troubleshooting
- **OOM** → keep `JAIS_ASR_MODEL=auto` (AWQ ~3.4 GB) and lower `JAIS_LLM_MAX_NEW_TOKENS`.
- **Slow replies on T4** → expected with the quantized 6B ASR (~3-8 s per turn).
- **`TTS emitted no speech tokens`** → set `JAIS_TTS_REF_WAV` + `JAIS_TTS_REF_TEXT` (custom voice files).
- **Browser can't connect** → first Connect warms models (~1-2 min); retry, and only use the `https://…proxy.googlers.com` URL Colab prints.
- أداة إضافية: `!tail -30 /content/jais-voice.log` لعرض سجلات الخادم مباشرة.